
# LAB 07

### TITLE: RNN Encoder-Decoder for Statistical Machine Translation

#### OBJECTIVE:

To develop an RNN Encoder–Decoder model for Statistical Machine Translation (SMT) and investigate how recurrent neural networks learn the mapping between source and target language sequences.

### THEORY:

The Encoder–Decoder model is a deep learning architecture specifically designed to address Sequence-to-Sequence (Seq2Seq) tasks, where both the input and output are sequences and may have different lengths.

Unlike conventional neural networks that generate outputs of a fixed size, the Encoder–Decoder model produces the output sequence one element at a time. This sequential generation capability makes it highly suitable for a variety of applications, including:

Machine Translation

Text Summarization

Chatbots

Speech Recognition

Question Answering

The architecture commonly employs Recurrent Neural Networks (RNNs) or Long Short-Term Memory (LSTM) networks, as these models are particularly effective at processing and learning patterns from sequential data.

### Statistical Machine Translation (SMT)
 It is a machine learning approach that translates text from one language to another using statistical models learned from large bilingual datasets. Traditional SMT systems rely on phrase tables and probability distributions, which often fail to capture the semantic meaning and long-range dependencies between words.

The RNN Encoder–Decoder model was introduced to overcome these limitations by learning continuous vector representations of sentences. It uses two Recurrent Neural Networks (RNNs): an Encoder, which converts the input sentence into a fixed-length vector, and a Decoder, which generates the translated sentence from this vector. This architecture improves translation quality by learning contextual and semantic relationships directly from data.

#### Encoder

The Encoder is the first part of the model.

Its function is to:

1. Read the complete input sequence one element at a time.

2. Learn important information from the sequence.

3. Convert the sequence into a fixed-length vector called the Context Vector.

4. Pass the final hidden state and cell state to the decoder

#### Context Vector

The Context Vector is the compressed representation of the entire input sequence.

It contains:

Semantic information

Sequence information

Learned patterns

The decoder uses this vector to generate the output sequence.

#### Decoder

The Decoder receives the Context Vector from the Encoder.

Its functions are:

1. Start with a special Start-of-Sequence (SOS) token.

2. Predict one output token at a time.

3. Use the previous predicted word to predict the next word.

4. Continue until the End-of-Sequence (EOS) token is generated.

#### Working of Encoder–Decoder Model

The model works in the following steps:

1. Input sequence is given to the Encoder.

2. Encoder processes each input token using LSTM.

3. Final hidden state and cell state become the Context Vector.

4. Decoder receives these states as its initial state.

5. Decoder predicts one output token at each time step.

6. The prediction continues until the End-of-Sequence token is produced.

----------------------------------------

In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data



In [2]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

 --> SOS (Start of Sentence) and EOS (End of Sentence) tokens are defined which creates the Lang class to store vocabulary information.

Also it assigns an integer index to each word, counts word frequency.And, 

Creates mappings:

word → index

index → word

In [3]:
# Turn a Unicode string to plain ASCII, thanks to

def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

This cell cleans the dataset. The following functions are performed here:

Convert Unicode → ASCII

Convert uppercase → lowercase

Remove unnecessary symbols

Only letters and punctuation are kept

In [4]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

This function loads the translation dataset.

Steps:

1. Read file.
2. Split into lines.
3. Separate English and French sentences.
4. Normalize text.
5. Reverse language pairs (French → English).
6. Create language vocabularies.

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.


In [9]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

Here, not every sentence is used.

This cell removes long and complex sentences.

Now, let's bundle everything up as per the diagram above: 

### Data Preparation

In [5]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [10]:
PATH = r'data\eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['je suis libre', 'i m free']


15

--> the English–French dataset is loaded.

`Note`: Here, we have only inserted lower case in the word2index, therefore, use of uppercase letters will yield an error. 

## Encoder-Decoder Network

### Encoder RNN

In [11]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

Here, we convert the input word indices into dense embedding vectors. During training, these embeddings are learned and gradually capture the semantic relationships and meanings of the words.

Functions:

-Convert word index into embedding vector.

-Process one word after another.

-Produce final hidden state (context vector).

"The encoder does not generate translated words."


### Decoder RNN

In [12]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1) # values, index of the highest-scoring word
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input (removes the last dimension before detaching)

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, hidden):
        output = self.embedding(input)
        output = F.relu(output)
        output, hidden = self.rnn(output, hidden)
        output = self.out(output)
        return output, hidden

-> The steps involved are:

1. Receive Encoder hidden state.

2. Start with SOS token.

3. Predict first word.

4. Feed predicted word back.

5. Predict next word.

6. Stop at EOS token.

## Training



## Prepare Training Data

For each pair:
-  we will need an input tensor (indexes of the words in the input sentence) and 
- target tensor (indexes of the words in the target sentence). 
- While creating these vectors we will append the EOS token to both sequences.

In [13]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [15]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

This performs one complete training iteration.

Steps:

1. Read batch.
2. Encoder forward pass.
3. Decoder forward pass.
4. Compute loss.
5. Backpropagation.
6. Update weights using Adam optimizer.

In [16]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

--> Useful for monitoring long training sessions.

In [18]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [19]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [20]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [21]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [22]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = DecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 7s (- 5m 5s) (5 2%) 1.9605
0m 17s (- 5m 40s) (10 5%) 1.2822
0m 27s (- 5m 34s) (15 7%) 1.0993
0m 32s (- 4m 53s) (20 10%) 0.9779
0m 37s (- 4m 22s) (25 12%) 0.8658
0m 42s (- 3m 59s) (30 15%) 0.7645
0m 47s (- 3m 41s) (35 17%) 0.6733
0m 52s (- 3m 30s) (40 20%) 0.5922
0m 57s (- 3m 18s) (45 22%) 0.5181
1m 2s (- 3m 7s) (50 25%) 0.4549
1m 7s (- 2m 57s) (55 27%) 0.3989
1m 14s (- 2m 54s) (60 30%) 0.3527
1m 29s (- 3m 6s) (65 32%) 0.3066
1m 41s (- 3m 7s) (70 35%) 0.2694
1m 46s (- 2m 57s) (75 37%) 0.2400
1m 52s (- 2m 48s) (80 40%) 0.2094
1m 57s (- 2m 38s) (85 42%) 0.1870
2m 2s (- 2m 29s) (90 45%) 0.1639
2m 6s (- 2m 20s) (95 47%) 0.1528
2m 11s (- 2m 11s) (100 50%) 0.1386
2m 16s (- 2m 3s) (105 52%) 0.1245
2m 21s (- 1m 55s) (110 55%) 0.1143
2m 25s (- 1m 47s) (115 57%) 0.1071
2m 30s (- 1m 40s) (120 60%) 0.1002
2m 35s (- 1m 33s) (125 62%) 0.0935
2m 40s (- 1m 26s) (130 65%) 0.092

In [23]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je suis cultive
= i m cultured
< he s very ill <EOS>

> je suis tres gras
= i m very fat
< i m very fat <EOS>

> tu es confus
= you re confused
< i m not crying <EOS>

> il est scenariste
= he s a scriptwriter
< they re babies <EOS>

> tu es fort effronte
= you re very forward
< you re very forward <EOS>

> je suis au lit
= i m in bed
< i m in bed <EOS>

> je suis veinarde
= i m lucky
< he s a gambler <EOS>

> ils sont extenues
= they are exhausted
< i m happy here <EOS>

> vous etes charmantes
= you re charming
< you re all happy <EOS>

> vous etes mince
= you re thin
< you re all alone <EOS>



### DISCUSSION

In this experiment, an RNN Encoder–Decoder model was implemented for Statistical Machine Translation (SMT) using PyTorch. The English–French dataset was preprocessed to improve data quality through text normalization, cleaning, and sentence filtering.

The Encoder converted the input sentence into a context vector, while the Decoder generated the translated sentence word by word. During training, the model minimized translation loss using the Adam optimizer, and the decreasing loss indicated improved learning.

The model successfully learned sequence-to-sequence translation, although its performance may decline for longer sentences because it relies on a fixed-length context vector. Overall, the experiment provided practical understanding of Encoder–Decoder architectures, recurrent neural networks, and neural machine translation.
